# Phase 1 & 2 : Prétraitement et PNL Classique (TF-IDF + Régression Logistique / Naive Bayes)

Ce notebook présente le prétraitement des données textuelles de Twitter et la mise en œuvre de modèles statistiques classiques pour l'analyse des sentiments de la plateforme **BrandPulse AI**.

In [ ]:
import os
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter

# Ajout du dossier parent au path pour importer preprocessing
import sys
sys.path.append('..')
from preprocessing import clean_tweet

print("Importations terminées avec succès.")

## 1. Collecte et Chargement des Données

Nous téléchargeons le jeu de données **Twitter US Airline Sentiment** depuis un dépôt public GitHub.

In [ ]:
# Création des dossiers nécessaires
os.makedirs('../data', exist_ok=True)
os.makedirs('../models', exist_ok=True)

data_path = '../data/Tweets.csv'
urls = [
    'https://raw.githubusercontent.com/satyajeetkrjha/kaggle-Twitter-US-Airline-Sentiment-/master/Tweets.csv',
    'https://raw.githubusercontent.com/satyajeetkrjha/kaggle-Twitter-US-Airline-Sentiment-/refs/heads/master/Tweets.csv',
    'https://raw.githubusercontent.com/kolaveridi/kaggle-Twitter-US-Airline-Sentiment-/master/Tweets.csv'
]

if not os.path.exists(data_path):
    print("Téléchargement du jeu de données en cours...")
    success = False
    for url in urls:
        try:
            print(f"Tentative de téléchargement depuis : {url}")
            urllib.request.urlretrieve(url, data_path)
            print("Téléchargement terminé !")
            success = True
            break
        except Exception as e:
            print(f"Échec : {e}")
    if not success:
        raise RuntimeError("Impossible de télécharger le dataset depuis les sources.")
else:
    print("Le fichier de données existe déjà.")

In [ ]:
# Lecture du fichier CSV
df = pd.read_csv(data_path)
print(f"Forme du dataset : {df.shape}")
print("\nAperçu des données :")
print(df[['airline_sentiment', 'text']].head())

## 2. Prétraitement linguistique (Nettoyage des Tweets)

Nous appliquons les fonctions définies dans `preprocessing.py` pour nettoyer chaque tweet (minuscules, retrait des mentions/liens, lemmatisation et retrait des mots vides).

In [ ]:
print("Nettoyage des tweets en cours (cette étape peut prendre 1 à 2 minutes)...")
df['clean_text'] = df['text'].apply(clean_tweet)

# Suppression des lignes qui sont vides après nettoyage
df = df[df['clean_text'].str.strip() != '']
print("Nettoyage terminé !")
print(df[['airline_sentiment', 'clean_text']].head())

## 3. Analyse Exploratoire des Données (EDA)

Voyons comment se répartissent les sentiments et quels mots reviennent le plus souvent.

In [ ]:
# Distribution des sentiments
plt.figure(figsize=(8, 5))
sns.countplot(x='airline_sentiment', data=df, order=['positive', 'neutral', 'negative'], palette='viridis')
plt.title('Distribution des Sentiments')
plt.xlabel('Sentiment')
plt.ylabel('Nombre de tweets')
plt.savefig('../data/sentiment_distribution.png', bbox_inches='tight')
plt.show()

print(df['airline_sentiment'].value_counts(normalize=True))

In [ ]:
# Génération des Nuages de Mots (Word Clouds) pour chaque sentiment
sentiments = ['positive', 'neutral', 'negative']
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, sent in enumerate(sentiments):
    text = " ".join(df[df['airline_sentiment'] == sent]['clean_text'])
    wordcloud = WordCloud(width=800, height=800, background_color='white', max_words=80).generate(text)
    axes[i].imshow(wordcloud, interpolation='bilinear')
    axes[i].set_title(f"Sentiment : {sent.capitalize()}", fontsize=16)
    axes[i].axis('off')

plt.suptitle('Nuages de mots par Sentiment', fontsize=20)
plt.savefig('../data/wordclouds.png', bbox_inches='tight')
plt.show()

## 4. Vectorisation TF-IDF & Séparation des Données

In [ ]:
from sklearn.model_selection import train_test_split

X = df['clean_text']
y = df['airline_sentiment']

# Séparation stratifiée en train/test (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Taille Train : {X_train.shape[0]}, Taille Test : {X_test.shape[0]}")

## 5. Entraînement des Modèles Classiques

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import joblib

# 1. Régression Logistique
lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

print("Entraînement de la Régression Logistique en cours...")
lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

print("\n=== RAPPORT : RÉGRESSION LOGISTIQUE ===")
print(classification_report(y_test, y_pred_lr))

In [ ]:
# 2. Naive Bayes
nb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ('clf', MultinomialNB())
])

print("Entraînement de Naive Bayes en cours...")
nb_pipeline.fit(X_train, y_train)
y_pred_nb = nb_pipeline.predict(X_test)

print("\n=== RAPPORT : NAIVE BAYES ===")
print(classification_report(y_test, y_pred_nb))

## 6. Analyse des Résultats & Matrices de Confusion

In [ ]:
# Visualisation des matrices de confusion
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr, ax=axes[0], cmap='Blues')
axes[0].set_title('Régression Logistique')

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_nb, ax=axes[1], cmap='Greens')
axes[1].set_title('Naive Bayes')

plt.tight_layout()
plt.savefig('../data/confusion_matrices_classical.png', bbox_inches='tight')
plt.show()

## 7. Sauvegarde du Meilleur Modèle Classique

In [ ]:
# La régression logistique offre généralement de meilleures performances globales pour TF-IDF.
joblib.dump(lr_pipeline, '../models/classical_pipeline.pkl')
print("Modèle Régression Logistique + TF-IDF sauvegardé dans '../models/classical_pipeline.pkl'.")